# FER-CE — Démo complète (Google Colab)

**Reconnaissance des Expressions Faciales Composées** sur RAF-DB (11 classes, numérotées 1..11).

Ce notebook exécute **tout le projet** de bout en bout :

| Étape | Couche | Ce qui se passe |
|------:|:------:|-----------------|
| 1 |  —  | Préparation Colab : GPU, Drive, dézippage, dépendances |
| 2 |  1  | Audit du dataset + cache des visages |
| 3 |  2  | **Sweep multi-modèles** (preset Colab : 5-7 runs, ~1-2 h sur T4) |
| 4 |  2  | Sélection du meilleur modèle + inférence individuelle |
| 5 |  2  | **Vision-LLM** Qwen2-VL — explication zero-shot |
| 6 |  3  | **Grad-CAM** + cohérence image/texte |
| 7 |  —  | **Rapport final pour présentation** (graphes + analyse) |

> **Architecture du projet.** Scripts organisés en `couche1/` (préparation données), `couche2/` (entraînement + explication LLM) et `couche3/` (interprétation Grad-CAM). Tout passe par `config.py`. Voir `explanation.md` pour la doc complète, `README.md` pour le run local, `README_COLAB.md` pour ce notebook.

> **Sans labels sémantiques.** Les classes sont désignées par leur **numéro** (1..11) — l'entraînement n'utilise aucun nom d'émotion. Seul le Vision-LLM, en zero-shot, parle d'émotions.

## 1.  Préparation de l'environnement

In [ ]:
# 1.1  GPU disponible ?
!nvidia-smi -L || echo "⚠ Aucun GPU détecté — passe en CPU (très lent)"

In [ ]:
# 1.2  Monter Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1.3  Chemin du zip projet sur Drive
# Le zip contient : couche1/, couche2/, couche3/, config.py, main.py, run_local.py,
# sweep.py, requirements.txt, ce notebook, README*.md, explanation.md
# + les sous-zips Image.zip, Annotation.zip, EmoLabel.zip (données RAF-DB).
DRIVE_ZIP_PATH = "/content/drive/MyDrive/deep_learning.zip"   # <-- adapte si besoin
PROJECT_DIR     = "/content/deep_learning"

In [ ]:
# 1.4  Dézippage du projet et des données
import os, shutil, zipfile

if os.path.isdir(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
os.makedirs(PROJECT_DIR, exist_ok=True)

with zipfile.ZipFile(DRIVE_ZIP_PATH) as zf:
    zf.extractall(PROJECT_DIR)

for sub in ("Image.zip", "Annotation.zip", "EmoLabel.zip"):
    path = os.path.join(PROJECT_DIR, sub)
    if os.path.exists(path):
        with zipfile.ZipFile(path) as zf:
            zf.extractall(PROJECT_DIR)
        os.remove(path)
        print(f"  [\u2713] {sub} dézippé")

for name in ("Image", "Annotation", "EmoLabel"):
    outer = os.path.join(PROJECT_DIR, name)
    inner = os.path.join(outer, name)
    if os.path.isdir(inner):
        for child in os.listdir(inner):
            shutil.move(os.path.join(inner, child), os.path.join(outer, child))
        os.rmdir(inner)

print("\nContenu de", PROJECT_DIR)
for entry in sorted(os.listdir(PROJECT_DIR)):
    full = os.path.join(PROJECT_DIR, entry)
    tag  = "DIR" if os.path.isdir(full) else "   "
    print(f"  [{tag}] {entry}")

os.chdir(PROJECT_DIR)
print("\ncwd =", os.getcwd())

In [ ]:
# 1.5  Dépendances (Colab fournit déjà torch/torchvision/sklearn)
# Pillow<12 pour la compat Qwen2-VL
!pip install -q -U transformers accelerate bitsandbytes timm seaborn "pillow<12"

In [ ]:
# 1.6  Vérifie que tous les scripts s'importent (nouveau layout par couche)
import sys, importlib
sys.path.insert(0, PROJECT_DIR)

for m in [
    "config",
    # Couche 1
    "couche1.face_detection", "couche1.augmentation", "couche1.normalization",
    "couche1.check_dataset", "couche1.data_exploration", "couche1.preprocessing",
    # Couche 2
    "couche2.dataset", "couche2.model", "couche2.training_utils",
    "couche2.train", "couche2.evaluate", "couche2.vision_llm",
    "couche2.explain", "couche2.test_model",
    # Couche 3
    "couche3.interpret",
    # Orchestrateurs
    "main", "run_local", "sweep",
]:
    importlib.import_module(m)
    print(f"  [OK] {m}")

import config
print(f"\nConfig clé :")
print(f"  NUM_CLASSES = {config.NUM_CLASSES} (numéros 1..{config.NUM_CLASSES})")
print(f"  MODELS      = {config.MODELS}")
print(f"  BATCH_SIZE  = {config.BATCH_SIZE}")
print(f"  USE_MIXUP={config.USE_MIXUP}  USE_CUTMIX={config.USE_CUTMIX}  USE_EMA={config.USE_EMA}")

## 2.  Couche 1 — Audit du dataset

Distribution des **numéros de classe** (1..11) et vérifications d'intégrité. Aucune sémantique : chaque image est identifiée par son numéro — l'esprit du *self-supervised learning* pour la partie reconnaissance.

In [ ]:
from couche1 import data_exploration
data_exploration.main()

In [ ]:
from couche1 import check_dataset
check_dataset.main()

In [ ]:
# Construit le cache des visages recadrés (~10 s à la 1re exécution, instantané ensuite)
from couche2.dataset import FERDataset
train_ds = FERDataset("train", train=True)
test_ds  = FERDataset("test",  train=False)
print(f"  train = {len(train_ds)} images, test = {len(test_ds)} images")
print(f"  classes train -> {train_ds.class_counts().tolist()}")

In [ ]:
from IPython.display import Image
for f in ("class_distribution.png", "sample_images.png"):
    path = os.path.join(config.OUTPUT_DIR, f)
    if os.path.exists(path):
        display(Image(filename=path))

## 3.  Couche 2 — Sweep multi-modèles (preset Colab)

On lance le **preset Colab** : 6 runs courts (10 époques chacun) avec des hyperparamètres différents :

1. `resnet50_baseline` — config par défaut
2. `resnet50_no_mix` — sans MixUp ni CutMix (sur-apprentissage)
3. `resnet50_focal` — Focal-Loss au lieu de CrossEntropy lissée
4. `resnet50_no_ema` — sans la moyenne mobile des poids
5. `vit_baseline` — Vision Transformer
6. `swin_baseline` — Swin Transformer

Chaque run est entraîné, évalué, **enregistré dans `outputs/registry.json`**, puis comparé. Compte ~1-2 h sur T4 gratuit. Pour aller plus vite tu peux n'en lancer qu'un sous-ensemble (voir cellule après).

In [ ]:
# Aperçu du preset Colab (et du preset local pour info) — sans rien lancer
import sweep
sweep.list_presets()

In [ ]:
# Lance les 6 runs du preset Colab (long).
# Pour ne lancer qu'une partie : remplace la ligne ci-dessous par
#     runs = [s for s in sweep.SWEEP_COLAB if s['run_name'] in {'resnet50_baseline', 'resnet50_focal'}]
import importlib, sweep
importlib.reload(sweep)

runs = sweep.SWEEP_COLAB
results = [sweep.run_one(s) for s in runs]
sweep.write_report(results, preset="colab")

In [ ]:
# Affiche les graphiques de comparaison
from IPython.display import Image as IPImage
for f in ("sweep_comparison.png", "sweep_curves.png"):
    p = os.path.join(config.OUTPUT_DIR, f)
    if os.path.exists(p):
        display(IPImage(filename=p))

## 4.  Couche 2 — Sélection du meilleur modèle + inférence

On lit la registry pour identifier le meilleur run, puis on l'utilise pour quelques inférences individuelles avec top-3.

In [ ]:
import json, run_local
run_local.list_models()

with open(run_local.REGISTRY_PATH) as f:
    reg = json.load(f)
BEST_RUN = max(reg["models"], key=lambda k:
               reg["models"][k].get("metrics", {}).get("accuracy", {}).get("test", 0))
BEST_ARCH = reg["models"][BEST_RUN].get("arch", BEST_RUN)
print(f"\n  -> Meilleur run : {BEST_RUN}   (arch = {BEST_ARCH})")

In [ ]:
import random, importlib
from couche2 import test_model
importlib.reload(test_model)

model, info = run_local.load_registered_model(BEST_RUN, prefer_ema=True)

imgs = [f for f in os.listdir(config.IMAGE_DIR) if f.startswith("test")][:6]
random.seed(0)
for i, name in enumerate(random.sample(imgs, min(6, len(imgs)))):
    test_model.test_one_image(os.path.join(config.IMAGE_DIR, name),
                              model, mode="bbox", index=i, show=True)

## 5.  Couche 2 — Vision-LLM (Qwen2-VL, zero-shot)

Un modèle vision-langage **non entraîné** sur cette tâche décrit l'émotion composée et **explique** les indices faciaux (sourcils, yeux, bouche…). Le numéro de classe vient du classifieur ; l'explication vient du Vision-LLM.

In [ ]:
from couche2 import explain
import importlib; importlib.reload(explain)

sample = [os.path.join(config.IMAGE_DIR, f)
          for f in os.listdir(config.IMAGE_DIR) if f.startswith("test")][:5]
explanations = explain.explain_images(sample, model_name=BEST_ARCH)

In [ ]:
from IPython.display import Image as IPImage
for i in range(len(explanations)):
    p = os.path.join(config.OUTPUT_DIR, f"explain_{i}.png")
    if os.path.exists(p): display(IPImage(filename=p))

## 6.  Couche 3 — Interprétation Grad-CAM + cohérence Vision-LLM

- **Grad-CAM** : où le CNN regarde dans le visage.
- **Découpage en 3 bandes** (front/sourcils, yeux, bouche/joues) — approximation des *Action Units*.
- **Cohérence** *Causal Emotion Grounding* : la zone activée correspond-elle aux indices cités par le LLM ?

In [ ]:
from couche3 import interpret
import importlib; importlib.reload(interpret)

interp = interpret.interpret_images(sample, model_name=BEST_ARCH,
                                    use_vlm=True, prefer_ema=True)

In [ ]:
from IPython.display import Image as IPImage
for i in range(len(interp)):
    p = os.path.join(config.OUTPUT_DIR, f"interpret_{i}.png")
    if os.path.exists(p): display(IPImage(filename=p))

## 7.  📊 Rapport final — présentation

Cette dernière cellule rassemble **tout ce qu'on a vu** dans un format présentable pour la classe :

- **Tableau** de tous les runs du sweep (classés par accuracy test)
- **Graphes** de comparaison + courbes superposées
- **Analyse run par run** (sur-/sous-apprentissage, divergence, etc.)
- **Synthèse « pourquoi certains runs dégradent »** — utile pour expliquer au prof

Le rapport est écrit dans `outputs/sweep_report.md` ; on l'affiche aussi inline.

In [ ]:
from IPython.display import Markdown, Image as IPImage
report_path = os.path.join(config.OUTPUT_DIR, "sweep_report.md")
with open(report_path) as f:
    md = f.read()

# Remplace les chemins relatifs des images par des chemins absolus pour Colab
md = md.replace("](sweep_comparison.png)",
                f"]({config.OUTPUT_DIR}/sweep_comparison.png)")
md = md.replace("](sweep_curves.png)",
                f"]({config.OUTPUT_DIR}/sweep_curves.png)")
display(Markdown(md))

In [ ]:
# En complément : les graphes du meilleur run (matrice de confusion, par classe, global)
from IPython.display import Image as IPImage
for suffix in ("_overall", "_per_class", "_confusion"):
    p = os.path.join(config.OUTPUT_DIR, f"eval_{BEST_RUN}{suffix}.png")
    if os.path.exists(p):
        print(f"=== {os.path.basename(p)} ===")
        display(IPImage(filename=p))

## 8.  💾 Sauvegarde sur Drive (optionnel)

Le runtime Colab efface tout à la fin de la session. Cette cellule copie le dossier `outputs/` (poids, registry, rapport, graphes) vers Drive pour le conserver.

In [ ]:
# Décommente pour sauvegarder
# OUT_DRIVE = "/content/drive/MyDrive/fer_ce_outputs"
# import shutil
# os.makedirs(OUT_DRIVE, exist_ok=True)
# for f in os.listdir(config.OUTPUT_DIR):
#     src = os.path.join(config.OUTPUT_DIR, f)
#     if os.path.isfile(src):
#         shutil.copy(src, OUT_DRIVE)
# print('  [OK] outputs/ copié vers', OUT_DRIVE)